In [1]:
# Import Libraries
import pandas as pd
from bs4 import BeautifulSoup
import requests
import smtplib
from datetime import time, datetime, timedelta
import random, re

# Global Variables
global bullet_characters 
global salary_descriptors 
bullet_characters = ['-', '•', '◦', '▪', '▸', '➤', '★', '*']
salary_descriptors = ['$','pay','paid','salary','compensation']

In [2]:
# Extract HTML - Paginated Func
def extract(page):    
    url = f'https://www.teamworkonline.com/jobs-in-sports?employment_opportunity_search%5Bsort_by%5D=most_recent&page={page}'
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"]
    
    headers = {'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

In [3]:
# Extract HTML - Inner Link (a tag) Func
def extract_inner(url):    
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"]
    
    headers = {'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

In [4]:
def transform(jobs_soup):
    all_jobs = jobs_soup.find_all(class_ = "browse-jobs-card")
    raw_jobs = []
    for job in all_jobs:
##################################################################### Scraping Basic info from Job cards ############################################################################
        try:
            raw_job={
                'logo_image': job.find('img').get('src') if job.find('img') else None,
                'job_url': 'https://www.teamworkonline.com' + job.find('a').get('href') if job.find('a') else None,
                'title': job.find(class_ = "margin-none").get_text().strip() if job.find(class_ = "margin-none") else None,
                'company': job.find(class_ = "browse-jobs-card__content--organization").get_text().strip() if job.find(class_ = "browse-jobs-card__content--organization") else None,
                'location_info': job.find(class_ = "trending__content--small").get_text().strip() if job.find(class_ = "trending__content--small") else None,
                'experience_level': job.find(class_ = "browse-jobs-card__content--small").get_text().strip() if job.find(class_ = "browse-jobs-card__content--small") else None  
            }
####################################################################### Extracting info from the individual job site #################################################################     
            job_soup = extract_inner(raw_job['job_url'])
            raw_job['salary_info'] = salary_fetcher(job_soup)
            raw_job['details'], raw_job['active_job'] = detail_fetcher(job_soup)
                       
################################################### Scraping time based info from Job cards and doing minor extractions  #############################################################    
            posting_number_list = job.find_all(class_ = "browse-jobs-card__scoreboard")
            if len(posting_number_list)>= 2:
                raw_job['posting_numbers'] = posting_number_list[0].get_text().strip() + posting_number_list[1].get_text().strip()
            else:
                raw_job['posting_numbers'] = None
            raw_job['posting_date_identifier'] = job.find(class_ = "trending__content--small trending__scoreboard--time margin-left--xx-small").get_text().strip().split(' ')[0]
            raw_job['scrape_datetime'] = datetime.now()
            
            jobs_list.append(raw_job)
            
        except Exception as e:
            print(f"Error during job scrape: {e}")
            continue
    return

In [5]:
# This is code to find salary in the top portion of the sit
def salary_fetcher(job_soup):
    salary_desc = None
    try:
        salary_flag = job_soup.find_all('div',class_ = 'margin-right--x-small')
        for i in range(0,len(salary_flag)):
            if any(descriptor in str(salary_flag[i]) for descriptor in salary_descriptors):
                salary_desc = salary_flag[i].get_text()
            else:
                pass
    except Exception as e:
        print(f"Error during job scrape: {e}")
    return(salary_desc)

In [6]:
####################################### THIS IS THE NEW APPROACH AS OF 3/17 ##############################################
#text = "This is a sample test to see what will happen to be honest: \n\n\n- This is bullet 1.\n\n\n\n- This is bullet 2.\n- This is bullet 3.\n- This is bullet 4.\n- This is bullet 5. \n This is extra text Now that Idk what will happen to."

def wrap_bullets(html_text):
    if(html_text.find('ul')):
        for ul in html_text.find_all('ul'):
            for li in ul:
                li.string = '- ' + li.get_text()
    else:
        pass
    
    text = html_text.get_text(separator = '\n')
    
    lines = text.split('\n')
    result = []
    in_list = False
    
    for line in lines:
        stripped_line = line.strip()
        
        if in_list and stripped_line == '':
            continue
    
        if stripped_line and stripped_line[0] in bullet_characters:
            if not in_list:
                preceding_line = next((line for line in reversed(result) if line.strip()), None)
                result.append(f'<p>{preceding_line}</p>')
                result.append('<ul>')
                in_list = True
            content = stripped_line[1:].strip()
            result.append(f'<li>{content}</li>')
        else:
            if in_list:
                result.append('</ul>')
                in_list = False
            result.append(stripped_line)
            
    if in_list:
        result.append('</ul>')
    
    final_result = '\n'.join(result)
    
    soup = BeautifulSoup(final_result,'html.parser')

    return(soup.find_all('p'))

In [10]:
def detail_fetcher(job_soup):
    try:
        job_inner = job_soup.find('div', class_='opportunity-preview__body')
        active_job_tag = (False if job_soup.find('div', class_ = "opportunity-preview__closed") else True)
        job_details = job_inner.find_all('p')
        
        if job_details:
            if job_inner.find('ul'):
                p_tag = True
            else:
                p_tag = False
        else:
            p_tag = False
              
        if (p_tag):
            pass
        else:
            job_details = wrap_bullets(job_inner)
        
        data = {}
        headers = []
        for section in job_details:
            
            if not section:
                continue
            
            details = []
                
            header_text = section.get_text()
            next_tag = section.find_next_sibling()
            
            if (next_tag and next_tag.name in ['ul', 'ol']):
                headers.append(header_text)
                items = next_tag.find_all('li')
                details = [item.get_text(strip=True) for item in items]
            
                data[header_text] = details 
                    
        
        # Convert to single-row DataFrame
        details_df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
    except:
        details_df = []
    return(details_df, active_job_tag)

In [11]:
# Will use this once deployed pages = (1,11)
pages = ((1,3))
jobs_list = []
for page in range(pages[0],pages[1]):
    recent_jobs_html = extract(page)
    transform(recent_jobs_html)

In [81]:
# Data Cleaning Section - I believe all of this functionally works so far. Pickup here 3/28!
#pd.set_option('display.max_colwidth', None)
team_work_df = pd.DataFrame(jobs_list)

team_work_df['city'] = team_work_df['location_info'].str.split('·').str[0].str.strip()
team_work_df['state'] = team_work_df['location_info'].str.split('·').str[1].str.strip()
team_work_df.loc[team_work_df['city'].str.contains("[,]"),'city'] = team_work_df['city'].str.split(',').str[0].str.strip()
team_work_df['job_id'] = team_work_df['job_url'].str.split('-').str[-1].str.strip()
team_work_df.loc[~team_work_df['posting_date_identifier'].str.endswith('s'),'posting_date_identifier'] = team_work_df['posting_date_identifier'] + ('s')

team_work_df['posting_timestamp'] = team_work_df.apply(
    lambda row: row['scrape_datetime'] - timedelta(**{row['posting_date_identifier']: int(row['posting_numbers'])}),
    axis=1)

#temp_df['title'].value_counts()

In [82]:
team_work_df

logo_image  \
0    https://cf-production.teamworkonline.com/uploads/public/thumb_86420987-43d0-49ad-8e25-be98867fe4b0.jpg   
1    https://cf-production.teamworkonline.com/uploads/public/thumb_bce4816b-f6b0-42d5-8363-5e053c23ed0c.png   
2    https://cf-production.teamworkonline.com/uploads/public/thumb_9092812b-025b-4c34-9542-1017299fbca2.png   
3    https://cf-production.teamworkonline.com/uploads/public/thumb_9092812b-025b-4c34-9542-1017299fbca2.png   
4    https://cf-production.teamworkonline.com/uploads/public/thumb_9092812b-025b-4c34-9542-1017299fbca2.png   
..                                                                                                      ...   
95   https://cf-production.teamworkonline.com/uploads/public/thumb_76159d88-681f-431e-a3fa-f26aa909f9fc.png   
96   https://cf-production.teamworkonline.com/uploads/public/thumb_037d0ccb-1507-433f-90b9-e17f3c28f0dd.png   
97  https://cf-production.teamworkonline.com/uploads/public/thumb_7c9c8cee-2294-4a5b-bef3-fb4f97342965.jpeg   
98  https://cf-production.teamworkonline.com/uploads/public/thumb_7c9c8cee-2294-4a5b-bef3-fb4f97342965.jpeg   
99  https://cf-production.teamworkonline.com/uploads/public/thumb_7c9c8cee-2294-4a5b-bef3-fb4f97342965.jpeg   

                                                                                                                                               job_url  \
0                                                https://www.teamworkonline.com/basketball-jobs/warriors/chase-center-jobs/stationary-engineer-2164287   
1                                             https://www.teamworkonline.com/hockey-jobs/sharkssports/san-jose-sharks/director-brand-marketing-2164286   
2   https://www.teamworkonline.com/arenas-facilities-jobs/oakview-group-/ovg-corporate/pastry-cook-iii-part-time-benchmark-international-arena-2164285   
3      https://www.teamworkonline.com/arenas-facilities-jobs/oakview-group-/ovg-corporate/premium-clubs-supervisor-part-time-allegiant-stadium-2164284   
4          https://www.teamworkonline.com/arenas-facilities-jobs/oakview-group-/ovg-corporate/bartender-part-time-fort-smith-convention-center-2164280   
..                                                                                                                                                 ...   
95                          https://www.teamworkonline.com/football-jobs/buffalo-bills/buffalo-bills-29559/seasonal-associate-guest-experience-2164192   
96          https://www.teamworkonline.com/multiple-properties/miller-sports-entertainment/saltlake-bees-jobs/maintenance-technician-full-time-2164190   
97                          https://www.teamworkonline.com/arenas-facilities-jobs/asm-global-jobs/asm-global-jobs/supervisor-food-and-beverage-2164184   
98                                       https://www.teamworkonline.com/arenas-facilities-jobs/asm-global-jobs/asm-global-jobs/warehouse-staff-2164186   
99                                      https://www.teamworkonline.com/arenas-facilities-jobs/asm-global-jobs/asm-global-jobs/cook-concessions-2164185   

                                                           title  \
0                                            Stationary Engineer   
1                                      Director, Brand Marketing   
2   Pastry Cook III  | Part-Time | Benchmark International Arena   
3       Premium Clubs Supervisor | Part-Time | Allegiant Stadium   
4           Bartender | Part-Time | Fort Smith Convention Center   
..                                                           ...   
95                         Seasonal Associate - Guest Experience   
96                            Maintenance Technician (Full-Time)   
97                                  Supervisor Food and Beverage   
98                                               Warehouse-staff   
99                                              Cook-Concessions   

            company           location_info experience_level  \
0      Chase Center      San Franc

In [110]:
team_work_df.loc[team_work_df['state'].str.len()>2,['title','location_info','city']]

,title,location_info,city
91,HR Business Partner,London · United Kingdom · Hybrid,London
98,"Major Gift Officer, West Coast",CA · Remote,CA
131,Event Operations + Merchandising Intern,FL · Hybrid,FL
132,Event Merchandising Manager (On-Site),FL · Hybrid,FL
134,Creative Strategist,United States · Remote,United States
135,Account Executive,United States · Remote,United States
231,Event Operations + Merchandising Intern,FL · Hybrid,FL
232,Event Merchandising Manager (On-Site),FL · Hybrid,FL
234,Creative Strategist,United States · Remote,United States
235,Account Executive,United States · Remote,United States


In [25]:
# Will use this once deployed pages = ((1,3),(3,5),(5,7),(7,9),(9,11))
pages = ((1,3))
for page_range in pages:
    for page_ind in range(page_range[0],page_range[1]):
        recent_jobs_html = extract(page_ind)
        transorm(recent_jobs_html)

1
2
3
4
5
6
7
8
9
10


In [12]:
### This is a copy of scaper code so I can split up some processing

all_jobs = soup.find_all(class_ = "browse-jobs-card")

for job in all_jobs:
##################################################################### Scraping Basic info from Job cards ####################################################
    
    logo_image = job.find('img').get('src')
    job_url = 'https://www.teamworkonline.com' + job.find('a').get('href')
    title = job.find(class_ = "margin-none").get_text().strip()
    company = job.find(class_ = "browse-jobs-card__content--organization").get_text().strip()
    location_info = job.find(class_ = "trending__content--small").get_text().strip()
    # Will need to split as such location_info.split('·')[0])
    experience = job.find(class_ = "browse-jobs-card__content--small").get_text().strip()
    
################################################### Scraping time based info from Job cards and doing minor calcualtions ####################################
    posting_number_list = job.find_all(class_ = "browse-jobs-card__scoreboard")
    posting_numbers = posting_number_list[0].get_text().strip() + posting_number_list[1].get_text().strip()
    posting_date_identifier = job.find(class_ = "trending__content--small trending__scoreboard--time margin-left--xx-small").get_text().strip().split(' ')[0]
    if posting_date_identifier.endswith('s'):
        pass
    else:
        posting_date_identifier = posting_date_identifier + ('s')
    posting_timestamp = datetime.now() - timedelta(**{posting_date_identifier: int(posting_numbers)})

'02/27/2026 12:58:28'

tests


In [20]:
######################### OLD CODE - N0 LONGER USED BUT KEPT BECAUSE ALOT OF WORK ##########################################

#.find_all(['p','strong']) •
data = {}
headers = []
for section in job_details:
    if (p_tag):
        full_text = section
        
        if section.find('strong'):
            header = section.find('strong').get_text()
            content = section.find_next_sibling()
        else:
            header = ''
            content = ''
    else:
        full_text = wrap_bullets(section.get_text(separator = '\n'))
        print(type(full_text))
    #for br in section.find_all('br'):
    #    br.replace_with("\n")
    
    #full_text = section.get_text(separator = '\n')
    if not full_text:
        continue
    
    #print(full_text)
    #header_text,details = extract_bullets_from_text(full_text,header,content)
    
    #if header_text and details:
    #    headers.append(header_text)
    #    data[header_text] = details
    #else:
    details = []
        
    header_text = full_text.get_text()
    next_tag = full_text.find_next_sibling()
    
    if (next_tag and next_tag.name in ['ul', 'ol']):
        headers.append(header_text)
        items = next_tag.find_all('li')
        details = [item.get_text(strip=True) for item in items]
    
        data[header_text] = details 
            

# Convert to single-row DataFrame
df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
print(df)


def extract_bullets_from_text(text,strong_header,strong_content):
    """Extract header and bullets from text"""
    try:
        if strong_content and strong_header:
            header = strong_header
            content = strong_content.get_text().strip()
        else:
            if ':' in text:
                sections = text.split(':')
                #print(sections)
                header = sections[0].strip()
                content = sections[1:]
            elif '\n' in text:
                sections = text.split('\n', 1)
                header = sections[0].strip()
                content = sections[1].strip() if sections[1].strip() != '' else header
                
                
        if header and content:
            bullets = re.split(r'[-•·*]\s*', content)
            
            if len(bullets) <= 1:
                bullets = re.split(r'-\s+', content)
            
            bullets = [b.strip() for b in bullets if b.strip()]
            
            if bullets:
                return (header, bullets)
        
    except:
        print('Error')
        pass
    
    return (None, None)

AttributeError: 'str' object has no attribute 'get_text'

In [33]:
job_soup = extract_inner('https://www.teamworkonline.com/hockey-jobs/hockeyjobs/nhl-league-office/royalty-analyst-2163333')

In [8]:
################## FOR NOW I HAVE THE DATA I BELEIVE I NEED. TIME TO TAKE TIME TO LAY OUT THE DATABASE SCHEMA #############

In [80]:
#temp_df['salary_info'].loc[temp_df['title'] == 'Senior Manager, Commercial Strategy & Operations']

In [9]:
active_job_tag

NameError: name 'active_job_tag' is not defined

In [181]:
headers

['The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:',
 'Required',
 'Preferred',
 'Education/Certifications',
 'Required Skills',
 'These core competencies reflect the underlying values that are necessary to represent the National Hockey League:']

In [182]:
# CODE TO PULL OFF RELEVANT INFO FROM DETAIL SCRAPE
df.filter(regex = '(?i)requ|respon|qual|look|skill|know|job|function|Elig|edu|exper|duties|duty',axis=1)

,The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:,Required,Education/Certifications,Required Skills
0,Processing statements in the royalty system,Advanced Microsoft Excel skills,"Minimum of Bachelor’s degree, Accounting or Fi...",Excellent verbal and written skills
1,Reviewing royalties reported for assigned lice...,NaN,NaN,Strong analytical and organizational skills an...
2,Analyzing revenue generated from royalty state...,NaN,NaN,Able to handle multiple tasks simultaneously
3,Reviewing and applying cash received from lice...,NaN,NaN,Take initiative and develop creative solutions...
4,Communicate with licensees regarding their roy...,NaN,NaN,NaN
5,Assisting with the preparation of royalty repo...,NaN,NaN,NaN
6,Assist with testing new releases of the royalt...,NaN,NaN,NaN
7,Ad hoc special projects,NaN,NaN,NaN


In [183]:
details_df = []
for header in headers:
    details_df.append({header: df[header].str.split('\n').explode(header)})
details_df = pd.DataFrame(details_df)
details_df

,The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:,Required,Preferred,Education/Certifications,Required Skills,These core competencies reflect the underlying values that are necessary to represent the National Hockey League:
0,0 Processing statements in the royalt...,NaN,NaN,NaN,NaN,NaN
1,NaN,0 Advanced Microsoft Excel skills 1 ...,NaN,NaN,NaN,NaN
2,NaN,NaN,0 Experience with consumer products licensi...,NaN,NaN,NaN
3,NaN,NaN,NaN,"0 Minimum of Bachelor’s degree, Accounting ...",NaN,NaN
4,NaN,NaN,NaN,NaN,0 Excellent verbal and writte...,NaN
5,NaN,NaN,NaN,NaN,NaN,0 Accountability 1 ...
